![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/data-preprocessing/SparkNLP_DocumentTitleSplitter_Demo.ipynb)

# DocumentTitleSplitter in Spark NLP

This notebook introduces `DocumentTitleSplitter` annotator that groups element level documents into semantic sections using `Title` metadata.

It is especially useful after `Reader2Doc` when you want to keep headings with their following content instead of chunking by raw size only.

In this notebook you will learn:

- how to configure `Reader2Doc` so the splitter can see the full ordered element stream
- what the default title aware output looks like
- how `explodeSplits`, `joinString`, and overflow splitting change the result
- what happens when the input has no `Title` metadata at all


## Why This Annotator Exists

Many unstructured document formats already contain semantic structure such as headings, paragraphs, and page boundaries. `Reader2Doc` can preserve that information in annotation metadata, for example:

- `elementType -> Title`
- `elementType -> NarrativeText`
- `pageNumber -> 1`

`DocumentTitleSplitter` uses that metadata to build section aware chunks.

The key rule is simple:

- when an input annotation has `metadata["elementType"] == "Title"`, it starts a new section
- the title stays with the following content
- optional overflow splitting can split very large sections afterward


In [1]:
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [6]:
import sparknlp

spark = sparknlp.start()
print("Spark version:", spark.version)

Spark version: 3.5.7


## Setup

We will use a small Markdown file from the Spark NLP test resources. The headings in that file become `Title` elements when `Reader2Doc` parses the content.

This notebook assumes you are using a Spark NLP version that already includes `DocumentTitleSplitter`.

In [7]:
import os
import urllib.request

from pyspark.ml import Pipeline
from pyspark.sql import functions as F

from sparknlp.annotator import DocumentTitleSplitter
from sparknlp.base import DocumentAssembler
from sparknlp.reader.reader2doc import Reader2Doc

base_url = "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/src/test/resources/reader"
demo_dir = os.path.abspath("document-title-splitter-demo")
os.makedirs(demo_dir, exist_ok=True)

sample_path = os.path.join(demo_dir, "title-chunking.md")
urllib.request.urlretrieve(f"{base_url}/md/title-chunking.md", sample_path)
sample_uri = f"file://{sample_path}"

empty_df = spark.createDataFrame([], "string").toDF("text")
print(sample_uri)

file:///content/document-title-splitter-demo/title-chunking.md


## Read the File as Element Level `DOCUMENT` Annotations

This is the most important part of the pipeline.

`DocumentTitleSplitter` needs the full ordered sequence of element level annotations in a single row. For that reason, `Reader2Doc` should be configured like this:

- `setOutputAsDocument(False)` to keep element level `DOCUMENT` annotations instead of merging everything into one document
- `setExplodeDocs(False)` so the ordered annotation array stays together in one row

If you explode the reader output into separate rows, the splitter loses the context it needs to form semantic sections.

In [8]:
reader2doc = (
    Reader2Doc()
    .setContentType("text/markdown")
    .setContentPath(sample_uri)
    .setOutputCol("document")
    .setOutputAsDocument(False)
    .setExplodeDocs(False)
)

reader_df = Pipeline(stages=[reader2doc]).fit(empty_df).transform(empty_df)

elements_df = (
    reader_df
    .select(F.posexplode("document").alias("idx", "ann"))
    .select(
        "idx",
        F.col("ann.result").alias("text"),
        F.col("ann.metadata")["elementType"].alias("elementType"),
        F.col("ann.metadata")["pageNumber"].alias("pageNumber"),
    )
)

elements_df.show(truncate=False)

+---+------------------------------------------------------------------------------------------------------------------------------------------+-------------+----------+
|idx|text                                                                                                                                      |elementType  |pageNumber|
+---+------------------------------------------------------------------------------------------------------------------------------------------+-------------+----------+
|0  |Overview                                                                                                                                  |Title        |1         |
|1  |Unstructured can parse Markdown into elements. This makes it easy to experiment with chunking locally.                                    |NarrativeText|1         |
|2  |Configuration                                                                                                                             |Title 

The `elementType` column is what drives the semantic split. In this example, the Markdown headings become `Title` elements, and the body text becomes `NarrativeText`.

## Default Title Aware Splitting

With the default configuration, `DocumentTitleSplitter` builds one chunk per title defined section.

Important defaults:

- `joinString = " "`
- `splitOnPageChange = False`
- `enableOverflowSplitting = False`
- `maxCharacters = 500` but it is only used when overflow splitting is enabled
- `explodeSplits = False`


In [9]:
default_splitter = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
)

default_df = Pipeline(stages=[reader2doc, default_splitter]).fit(empty_df).transform(empty_df)
default_splits = default_df.select("splits").first()["splits"]

for i, ann in enumerate(default_splits, start=1):
    print(f"Chunk {i}")
    print("sectionTitle:", ann.metadata.get("sectionTitle"))
    print("text:", ann.result)
    print("-" * 100)

Chunk 1
sectionTitle: Overview
text: Overview Unstructured can parse Markdown into elements. This makes it easy to experiment with chunking locally.
----------------------------------------------------------------------------------------------------
Chunk 2
sectionTitle: Configuration
text: Configuration max_characters controls hard chunk size. new_after_n_chars controls a softer preferred limit. combine_text_under_n_chars helps avoid tiny chunks.
----------------------------------------------------------------------------------------------------
Chunk 3
sectionTitle: Example
text: Example When the parser detects headings, they become Title elements. The chunk_by_title function uses those Title elements as section boundaries.
----------------------------------------------------------------------------------------------------


Each output annotation is still a `DOCUMENT`, but it now represents a semantic section instead of a raw reader element. The output metadata also includes `sectionTitle` when the section starts from a title.

## Explode Splits to Rows

When `explodeSplits=True`, each resulting chunk is placed on its own row. This is useful when you want downstream Spark stages to operate on one section per row.

In [10]:
exploded_splitter = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
    .setExplodeSplits(True)
)

exploded_df = Pipeline(stages=[reader2doc, exploded_splitter]).fit(empty_df).transform(empty_df)

exploded_df.selectExpr(
    "splits[0].metadata.sectionTitle as sectionTitle",
    "length(splits[0].result) as length",
    "splits[0].result as text",
).show(truncate=80)

+-------------+------+--------------------------------------------------------------------------------+
| sectionTitle|length|                                                                            text|
+-------------+------+--------------------------------------------------------------------------------+
|     Overview|   111|Overview Unstructured can parse Markdown into elements. This makes it easy to...|
|Configuration|   159|Configuration max_characters controls hard chunk size. new_after_n_chars cont...|
|      Example|   146|Example When the parser detects headings, they become Title elements. The chu...|
+-------------+------+--------------------------------------------------------------------------------+



## Customize How Elements Are Joined

By default, texts inside a section are joined with a single space. You can change that with `setJoinString`.

A common choice is `"\n\n"` when you want to preserve stronger visual separation between titles and paragraphs.

In [11]:
joined_splitter = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
    .setJoinString("\n\n")
)

joined_df = Pipeline(stages=[reader2doc, joined_splitter]).fit(empty_df).transform(empty_df)
print(joined_df.select("splits").first()["splits"][0].result)

Overview

Unstructured can parse Markdown into elements. This makes it easy to experiment with chunking locally.


## Overflow Splitting

Overflow splitting is a second pass that only runs after title based grouping.

The sequence is:

1. Build semantic sections from `Title` boundaries.
2. If `enableOverflowSplitting=True`, split only the sections that are larger than `maxCharacters`.

This means a large section can produce multiple chunks even though it still belongs to the same title defined section.

In [12]:
overflow_splitter = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
    .setEnableOverflowSplitting(True)
    .setMaxCharacters(80)
    .setExplodeSplits(True)
)

overflow_df = Pipeline(stages=[reader2doc, overflow_splitter]).fit(empty_df).transform(empty_df)

overflow_df.selectExpr(
    "splits[0].metadata.sectionTitle as sectionTitle",
    "length(splits[0].result) as length",
    "splits[0].result as text",
).show(truncate=80)

+-------------+------+------------------------------------------------------------------------------+
| sectionTitle|length|                                                                          text|
+-------------+------+------------------------------------------------------------------------------+
|     Overview|    77| Overview Unstructured can parse Markdown into elements. This makes it easy to|
|     Overview|    33|                                             experiment with chunking locally.|
|Configuration|    72|      Configuration max_characters controls hard chunk size. new_after_n_chars|
|Configuration|    78|controls a softer preferred limit. combine_text_under_n_chars helps avoid tiny|
|Configuration|     7|                                                                       chunks.|
|      Example|    73|     Example When the parser detects headings, they become Title elements. The|
|      Example|    72|      chunk_by_title function uses those Title elements as s

## Inputs Without `Title` Metadata

If the input does not contain `Title` metadata, `DocumentTitleSplitter` falls back to treating the whole input as a single section.

In that case:

- with overflow splitting disabled, you get one chunk
- with overflow splitting enabled, that single section can still be split by size

This makes the annotator safe to use even when title metadata is not available.

In [13]:
plain_text = (
    "Spark NLP can process long plain-text documents even when no title metadata is available. "
    "In that case DocumentTitleSplitter treats the full input as one semantic section. "
    "If overflow splitting is enabled, the annotator can still break the text into smaller chunks. "
    "This behavior is useful for simple text ingestion pipelines and for fallback processing."
)

plain_df = spark.createDataFrame([(plain_text,)], ["text"])

document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

plain_no_overflow = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
)

plain_with_overflow = (
    DocumentTitleSplitter()
    .setInputCols(["document"])
    .setOutputCol("splits")
    .setEnableOverflowSplitting(True)
    .setMaxCharacters(120)
)

plain_one_chunk_df = Pipeline(stages=[document_assembler, plain_no_overflow]).fit(plain_df).transform(plain_df)
plain_many_chunks_df = Pipeline(stages=[document_assembler, plain_with_overflow]).fit(plain_df).transform(plain_df)

plain_one_chunk = plain_one_chunk_df.select("splits").first()["splits"]
plain_many_chunks = plain_many_chunks_df.select("splits").first()["splits"]

print("Without overflow splitting:", len(plain_one_chunk), "chunk")
print("With overflow splitting:", len(plain_many_chunks), "chunks")

for i, ann in enumerate(plain_many_chunks, start=1):
    print(f"Chunk {i} ({len(ann.result)} chars): {ann.result}")

Without overflow splitting: 1 chunk
With overflow splitting: 4 chunks
Chunk 1 (102 chars): Spark NLP can process long plain-text documents even when no title metadata is available. In that case
Chunk 2 (116 chars): DocumentTitleSplitter treats the full input as one semantic section. If overflow splitting is enabled, the annotator
Chunk 3 (113 chars): can still break the text into smaller chunks. This behavior is useful for simple text ingestion pipelines and for
Chunk 4 (20 chars): fallback processing.


## Notes on Page Boundaries

`DocumentTitleSplitter` also supports `setSplitOnPageChange(True)`.

That option is most useful when your reader output includes reliable `pageNumber` metadata, such as PDF-derived content. It can be combined with title boundaries when you want sectioning to respect page transitions as well.

This notebook does not force a page-split example because the Markdown sample is already organized around titles, but the parameter is available when page-aware behavior is needed.

## Key Takeaways

- `DocumentTitleSplitter` is designed for semantic chunking from reader metadata, not raw text parsing.
- The recommended upstream source is `Reader2Doc().setOutputAsDocument(False).setExplodeDocs(False)`.
- `Title` elements define the primary section boundaries.
- `joinString`, `explodeSplits`, `splitOnPageChange`, and optional overflow splitting let you adapt the output to downstream indexing or LLM workflows.
- When no title metadata exists, the annotator still behaves predictably by treating the input as one section and optionally applying overflow splitting.
